In [ ]:
from datasets import load_dataset_builder, load_dataset, get_dataset_split_names, load_from_disk
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path
from huggingface_hub import hf_hub_download
import evaluate
from transformers import AutoImageProcessor
from torchvision.transforms import ColorJitter
import numpy as np
import torch
from torch import nn
from transformers import AutoModelForSemanticSegmentation, TrainingArguments, Trainer

import warnings
warnings.filterwarnings("ignore")

In [ ]:
datasets = load_from_disk("./data/scene_parse_150/")
datasets

In [ ]:
ds = datasets['train'].select(range(500))
ds = ds.train_test_split(test_size=0.2, seed=42)
ds

In [ ]:
index = np.random.randint(0, len(ds['train']))
ds['train'][index]

In [ ]:
def visualize_seg_mask(image, mask, title=None):
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(np.asarray(image))
    ax.imshow(np.asarray(mask), alpha=0.7)
    ax.set_title(title if title else "Segmentation Mask Overlay")
    
    plt.show()

visualize_seg_mask(ds['train'][index]['image'], ds['train'][index]['annotation'], title=ds['train'][index]['scene_category'])

In [ ]:
repo_id = "huggingface/label-files"
filename = "ade20k-id2label.json"
id2label = json.loads(Path(hf_hub_download(repo_id, filename, repo_type="dataset")).read_text())
id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

In [ ]:
train_ds = ds["train"]
test_ds = ds["test"]

In [ ]:
train_ds[index]['image']

In [ ]:
checkpoint = "nvidia/segformer-b0-finetuned-ade-512-512"
image_processor = AutoImageProcessor.from_pretrained(
    checkpoint,
    do_reduce_labels=True,
    size={"height": 256, "width": 256}   # resolusi lebih kecil dari default 512
)

In [ ]:
jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1)

In [ ]:
def train_transforms(example_batch):
    images = [jitter(x) for x in example_batch["image"]]
    labels = [x for x in example_batch["annotation"]]
    inputs = image_processor(images, labels)
    return inputs


def val_transforms(example_batch):
    images = [x for x in example_batch["image"]]
    labels = [x for x in example_batch["annotation"]]
    inputs = image_processor(images, labels)
    return inputs

In [ ]:
train_ds.set_transform(train_transforms)
test_ds.set_transform(val_transforms)

In [ ]:
metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)

        pred_labels = logits_tensor.detach().cpu().numpy()
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=num_labels,
            ignore_index=255,
            reduce_labels=False,
        )
        for key, value in metrics.items():
            if isinstance(value, np.ndarray):
                metrics[key] = value.tolist()
        return metrics

In [ ]:
model = AutoModelForSemanticSegmentation.from_pretrained(checkpoint, id2label=id2label, label2id=label2id)

In [ ]:
type(train_ds), type(test_ds)

In [ ]:
training_args = TrainingArguments(
    output_dir="segformer-b0",
    learning_rate=6e-5,
    num_train_epochs=1,            # cukup 1 dulu untuk tes
    max_steps=500,                 # batasi step biar cepat
    per_device_train_batch_size=1, # aman di RAM kecil
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    save_total_limit=1,            # hanya simpan checkpoint terakhir
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=500,                # simpan sekali di akhir
    eval_steps=500,                # evaluasi sekali di akhir
    logging_steps=50,
    eval_accumulation_steps=1,
    remove_unused_columns=False,
    push_to_hub=False,
    fp16=False,                    # matikan, karena di CPU/Windows sering bermasalah
    dataloader_num_workers=0       # aman di Windows, tidak error worker
)


trainer = Trainer(
    model=model, # model yang digunakan
    args=training_args, # argumen training
    train_dataset=train_ds, # dataset training
    eval_dataset=test_ds, # dataset evaluasi
    compute_metrics=compute_metrics, # fungsi untuk menghitung metriks
)


trainer.train()

In [ ]:
model.save_pretrained("models/segformer-b0")